In [15]:
import pandas as pd
import numpy as np

r_cols = ['use_id', 'movie_id', 'rating']
ratings = pd.read_csv("C:/Users/Matheus/Documents/LAMIA-Bootcamp/card-11/data/u.data", sep= '\t', names = r_cols, usecols=range(3))
ratings.head()

,use_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


In [ ]:
movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})
# agrupa pelo movie_id, e depois para cada grupo agrupado pelo movie_id conta quantas avaliações o fileme teve e a media de avaliação
movieProperties.head()

C:\Users\Matheus\AppData\Local\Temp\ipykernel_19684\2505249100.py:1: FutureWarning: The provided callable <function mean at 0x000002137F6EDF30> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})


rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

In [ ]:
movieNumRatings = pd.DataFrame(movieProperties['rating']['size'])
# transforma size em um dataframe 
movieNormalizeNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x)) / (np.max(x) - np.min(x)))
# utiliza a forma para normalizar a avaliação de cada filme, retornando numero de 0 a 1, sendo 1 maior avaliação
movieNormalizeNumRatings.head()

,size
movie_id,
1,452
2,131
3,90
4,209
5,86


In [ ]:
movieDict = {}
with open('C:/Users/Matheus/Documents/LAMIA-Bootcamp/card-11/data/u.item', encoding='latin-1') as f:
    # abre o arquivo e nomeia como f
    temp = ''
    for line in f:
        fields= line.rstrip('\n').split('|')
        # le ate a quebra de linhae separa por |
        movieID = int(fields[0])
        #pega a coluna 0
        name = fields[1]
        genrs = fields[5:25]
        #pega os generos, numerados com 1 ou 0
        genrs = list(map(int, genrs))
        # itera por todas as linhas
        movieDict[movieID] = (name, genrs, movieNormalizeNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating.get('mean'))
        # coloca no dicionario a quantidade de avaliações e a avaliação media 

In [30]:
movieDict[1]

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 np.float64(3.8783185840707963))

In [ ]:
from scipy import spatial

def ComputeDistance(a,b):
    genrsA = a[1]
    genrsB = b[1]
    genrsDistance = spatial.distance.cosine(genrsA,genrsB)
    popularityA = a[2]
    popularityB = b[2]
    popularityDistance = abs(popularityA - popularityB)
    return genrsDistance + popularityDistance

ComputeDistance(movieDict[2],movieDict[4])
# resumindo o que acontece, vai receber dois ids e calculará a diferença de genero e de avaliacao normalizada

np.float64(0.8004574042309892)

In [34]:
print(movieDict[2])
print(movieDict[4])

('GoldenEye (1995)', [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], np.float64(0.22298456260720412), np.float64(3.2061068702290076))
('Get Shorty (1995)', [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], np.float64(0.3567753001715266), np.float64(3.550239234449761))


In [ ]:
import operator

def getNeighbors(movieID, k):
    distances= []
    for movie in movieDict:
        if(movie !=movieID):
            dist = ComputeDistance(movieDict[movieID],movieDict[movie])
            distances.append((movie, dist))
    distances.sort(key=operator.itemgetter(1))
    neighbors = []
    for x in range(k):
        neighbors.append(distances[x][0])
    return neighbors
# esta função vai iterar sobre todos os filmes comparando se forem diferentes vai calcular a distancia e colocal na lista
K=10
# o numero de vixinhos que serao buscados
avRating = 0
neighbors = getNeighbors(1,K)
for neighbor in neighbors:
    avRating+= movieDict[neighbor][3]
    print(movieDict[neighbor][0]+ ' '+ str(movieDict[neighbor][3]))
# calcula a avaliação media de cada um dos 10 vizinhos
avRating /= float(K)
        

Liar Liar (1997)3.156701030927835
Aladdin (1992)3.8127853881278537
Willy Wonka and the Chocolate Factory (1971)3.6319018404907975
Monty Python and the Holy Grail (1974)4.0664556962025316
Full Monty, The (1997)3.926984126984127
George of the Jungle (1997)2.685185185185185
Beavis and Butt-head Do America (1996)2.7884615384615383
Birdcage, The (1996)3.4436860068259385
Home Alone (1990)3.0875912408759123
Aladdin and the King of Thieves (1996)2.8461538461538463


In [36]:
avRating

np.float64(3.3445905900235564)